# 07 | Export (CSV, GeoJSON and TopoJSON)

* **CSV:** the eight main tables, without geometry, to be imported into Power Query (Power BI).
* **GeoJSON and TopoJSON:** all layers, organised by territorial scope, to be used in the Shape Map and Deneb visuals. Each polygon layer is exported twice: the full-detail version and the simplified version created in notebook 06 (suffix `_simplified`).

Before the GeoJSON/TopoJSON export, the attribute columns that the map visuals do not need are removed, to keep the files small. The relationships are made in Power BI through the CSV tables.

## Setup

In [1]:
import sys
from pathlib import Path

# Make the src/ folder importable when running from the notebooks/ folder
sys.path.append(str(Path.cwd().parent))

import geopandas as gpd
import numpy as np
import pandas as pd

from src.utils import *

## CSV

In [2]:
csv_tables = {
    "postal_codes_PT_ES": "PostalCodes2024_PT_ES",
    "localities_PT_ES": "Localities2024_PT_ES",
    "nuts0_PT_ES": "NUTS0_2024_PT_ES",
    "nuts1_PT_ES": "NUTS1_2024_PT_ES",
    "nuts2_PT_ES": "NUTS2_2024_PT_ES",
    "nuts3_PT_ES": "NUTS3_2024_PT_ES",
    "districts_PT": "Districts2024_PT",
    "municipalities_PT": "Municipalities2024_PT",
}

csv_dir = OUTPUTS_DIR / "csv"
csv_dir.mkdir(parents=True, exist_ok=True)

for csv_name, layer_name in csv_tables.items():
    gdf = load_layer(layer_name)
    gdf.drop(columns=["geometry"]).to_csv(csv_dir / f"{csv_name}.csv", index=False)
    print(f"Exported '{csv_name}.csv'")

Exported 'postal_codes_PT_ES.csv'
Exported 'localities_PT_ES.csv'
Exported 'nuts0_PT_ES.csv'
Exported 'nuts1_PT_ES.csv'
Exported 'nuts2_PT_ES.csv'
Exported 'nuts3_PT_ES.csv'
Exported 'districts_PT.csv'
Exported 'municipalities_PT.csv'


## GeoJSON and TopoJSON

In [3]:
subfolders = {
    "PT_ES_WithArchipelagos": ["PostalCodes2024_PT_ES", "Localities2024_PT_ES", "NUTS0_2024_PT_ES",
                              "NUTS1_2024_PT_ES", "NUTS2_2024_PT_ES", "NUTS3_2024_PT_ES"],
    "PT_ES_Mainland": ["PostalCodes2024_PT_ES_mainland", "Localities2024_PT_ES_mainland", "NUTS0_2024_PT_ES_mainland",
                          "NUTS1_2024_PT_ES_mainland", "NUTS2_2024_PT_ES_mainland", "NUTS3_2024_PT_ES_mainland"],
    "PT_WithArchipelagos": ["Districts2024_PT", "Municipalities2024_PT", "PostalCodes2024_PT", "Localities2024_PT",
                           "NUTS1_2024_PT", "NUTS2_2024_PT", "NUTS3_2024_PT"],
    "PT_Mainland": ["Districts2024_PT_mainland", "Municipalities2024_PT_mainland", "PostalCodes2024_PT_mainland", "Localities2024_PT_mainland",
                       "NUTS1_2024_PT_mainland", "NUTS2_2024_PT_mainland", "NUTS3_2024_PT_mainland"],
    "ES_WithArchipelagos": ["PostalCodes2024_ES", "Localities2024_ES", "NUTS1_2024_ES", "NUTS2_2024_ES", "NUTS3_2024_ES"],
    "ES_Mainland": ["PostalCodes2024_ES_mainland", "Localities2024_ES_mainland", "NUTS1_2024_ES_mainland",
                       "NUTS2_2024_ES_mainland", "NUTS3_2024_ES_mainland"],
    "Archipelagos": ["PostalCodes2024_PT_archipelagos", "PostalCodes2024_ES_archipelagos"],
}

# Columns removed before the map export, by table type (the start of the layer name)
columns_to_drop = {
    "PostalCodes": ["NUTS3_ID", "NUTS2_ID", "NUTS1_ID", "NUTS0_ID", "LocalityID", "DegreeUrb"],
    "Localities": ["MunicipalityID", "DistrictID", "Population2024", "AreaKm2"],
    "NUTS3": ["MountainType", "UrbanType", "CoastalType"],
    "Districts": ["AreaHa", "PerimeterKm"],
    "Municipalities": ["AreaHa", "PerimeterKm"],
}

In [4]:
available_layers = set(gpd.list_layers(LAYERS_GPKG)["name"])


def prepare_for_map(gdf, layer_name):
    """Remove the columns the map visuals do not need and any empty geometry."""
    for prefix, cols in columns_to_drop.items():
        if layer_name.startswith(prefix):
            gdf = gdf.drop(columns=cols, errors="ignore")
    return gdf[~gdf.geometry.is_empty]


for subfolder, layer_names in subfolders.items():
    print(f"--- {subfolder}")
    for layer_name in layer_names:
        # Full-detail version
        export_geojson_topojson(prepare_for_map(load_layer(layer_name), layer_name), layer_name, subfolder)

        # Simplified version (only polygon layers have one)
        simplified_name = f"{layer_name}_simplified"
        if simplified_name in available_layers:
            export_geojson_topojson(prepare_for_map(load_layer(simplified_name), layer_name), simplified_name, subfolder)

--- PT_ES_WithArchipelagos
Saved 'PostalCodes2024_PT_ES' as GeoJSON
Saved 'PostalCodes2024_PT_ES' as TopoJSON
Saved 'Localities2024_PT_ES' as GeoJSON
Saved 'Localities2024_PT_ES' as TopoJSON
Saved 'Localities2024_PT_ES_simplified' as GeoJSON
Saved 'Localities2024_PT_ES_simplified' as TopoJSON
Saved 'NUTS0_2024_PT_ES' as GeoJSON
Saved 'NUTS0_2024_PT_ES' as TopoJSON
Saved 'NUTS0_2024_PT_ES_simplified' as GeoJSON
Saved 'NUTS0_2024_PT_ES_simplified' as TopoJSON
Saved 'NUTS1_2024_PT_ES' as GeoJSON
Saved 'NUTS1_2024_PT_ES' as TopoJSON
Saved 'NUTS1_2024_PT_ES_simplified' as GeoJSON
Saved 'NUTS1_2024_PT_ES_simplified' as TopoJSON
Saved 'NUTS2_2024_PT_ES' as GeoJSON
Saved 'NUTS2_2024_PT_ES' as TopoJSON
Saved 'NUTS2_2024_PT_ES_simplified' as GeoJSON
Saved 'NUTS2_2024_PT_ES_simplified' as TopoJSON
Saved 'NUTS3_2024_PT_ES' as GeoJSON
Saved 'NUTS3_2024_PT_ES' as TopoJSON
Saved 'NUTS3_2024_PT_ES_simplified' as GeoJSON
Saved 'NUTS3_2024_PT_ES_simplified' as TopoJSON
--- PT_ES_Mainland
Saved 'PostalCo

## File size comparison

In [5]:
geojson_dir = OUTPUTS_DIR / "geojson"

rows = []
for simplified_file in geojson_dir.rglob("*_simplified.geojson"):
    full_file = simplified_file.with_name(simplified_file.name.replace("_simplified", ""))
    full_mb = full_file.stat().st_size / 1_000_000
    simplified_mb = simplified_file.stat().st_size / 1_000_000
    rows.append({
        "layer": full_file.stem,
        "full_MB": round(full_mb, 2),
        "simplified_MB": round(simplified_mb, 2),
        "reduction_%": round(100 * (1 - simplified_mb / full_mb), 1),
    })

pd.DataFrame(rows).sort_values("full_MB", ascending=False).reset_index(drop=True)

,layer,full_MB,simplified_MB,reduction_%
0,Municipalities2024_PT,64.21,1.90,97.0
1,Municipalities2024_PT_mainland,49.70,1.67,96.6
2,Districts2024_PT,27.69,0.42,98.5
3,Districts2024_PT_mainland,16.02,0.29,98.2
4,Localities2024_PT_ES,14.14,13.53,4.3
5,Localities2024_PT_ES_mainland,13.74,13.14,4.4
6,Localities2024_ES,11.09,10.52,5.1
7,Localities2024_ES_mainland,10.86,10.30,5.2
8,Localities2024_PT,3.05,3.01,1.4
9,Localities2024_PT_mainland,2.88,2.84,1.4


## Summary
* Exported the eight main tables to CSV (without geometry) in `outputs/csv/`.
* For each layer, removed the columns that the map visuals do not need and any empty geometries.
* Exported every layer to GeoJSON and TopoJSON, organised in seven subfolders. Polygon layers are exported in two versions: full detail and simplified (`_simplified`).
  * **PT_ES_WithArchipelagos:** Portugal and Spain, including archipelagos
  * **PT_ES_Mainland:** Portugal and Spain, mainland only
  * **PT_WithArchipelagos:** Portugal, including archipelagos (districts, municipalities, postal codes, localities, NUTS 1–3)
  * **PT_Mainland:** mainland Portugal
  * **ES_WithArchipelagos:** Spain, including archipelagos
  * **ES_Mainland:** mainland Spain
  * **Archipelagos:** postal codes of the Portuguese and Spanish archipelagos

The `name` column required by the TopoJSON conversion is created from the index when it does not exist.

The last table compares the size of each full-detail GeoJSON file with its simplified version.